In [1]:
import os
import os.path
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
import os
import sys
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "test"  # Change to "test" for final submission

# Set to True to rebuild indices from CSV (required on first run)
# Set to False to load cached indices (faster for subsequent runs)
FORCE_REBUILD_INDICES = False

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/input/omnilex-indices")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"

# CSV corpus files for index building
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"

# Index cache paths
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"

# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Force rebuild indices: {FORCE_REBUILD_INDICES}")
print(f"\nCorpus files:")
print(f"  Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)" if LAWS_CSV.exists() else f"  Laws CSV: {LAWS_CSV} (NOT FOUND)")
print(f"  Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)" if COURTS_CSV.exists() else f"  Courts CSV: {COURTS_CSV} (NOT FOUND)")
print(f"\nIndex cache: {INDEX_PATH}")

Environment: Local
Dataset mode: test
Query file: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/test.csv
Validation mode: False
Force rebuild indices: False

Corpus files:
  Laws CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/laws_de.csv (73.0 MB)
  Courts CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/court_considerations.csv (2.43 GB)

Index cache: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed


# 2. Load Corpora and Build/Load Indices

In [3]:
from bm25index import BM25Index
import bm25index

In [4]:
# Load or build laws index
# Laws CSV: ~45MB, ~269K rows
# Build time: ~30 seconds | Load from cache: <1 second

laws_index = bm25index.get_or_build_index(
    name="laws",
    csv_path=LAWS_CSV,
    index_path=LAWS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    max_rows=100000000  # Uncomment to test with smaller corpus
)
print(f"\nLaws index: {len(laws_index.documents):,} documents")

# Test search
test_results = laws_index.search("Vertrag", top_k=3)
print(f"\nTest search 'Vertrag': {len(test_results)} results")
if test_results:
    print(test_results)

Loading cached laws index from /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed/laws_index.pkl
  Loaded 175,933 documents

Laws index: 175,933 documents


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Vertrag': 1 results
[{'query': 'vertrag', 'hits': [{'rank': 1, 'score': 2.8019, 'text': '4 wird ein rahmen vertrag mit nur einer anbieterin abgeschlossen, so wer den die auf die sem rahmen vertrag beruhenden einzel verträge entsprechend den bedingungen des rahmen vertrags abgeschlossen. für den abschluss der einzel verträge kann die auftraggeberin die jeweilige vertrags partnerin schriftlich auffordern, ihr ange bot zu vervollständigen.', 'citation': 'Art. 25 Abs. 4 BöB'}, {'rank': 2, 'score': 2.783, 'text': '6 ist das bakom registerbetreiberin, so unter steht der vertrag dem öffentlichen recht (verwaltungsrechtlicher vertrag); ist die auf gabe an einen dritten übertragen, so unter steht der vertrag dem privat recht (privatrechtlicher vertrag).', 'citation': 'Art. 17 Abs. 6 VID'}, {'rank': 3, 'score': 2.7276, 'text': '1 durch vertrag kann die verpflichtung zum abschluss eines künftigen vertrages begründet werden.', 'citation': 'Art. 22 Abs. 1 OR'}]}]


In [5]:
# Load or build courts index
# Courts CSV: ~2.3GB, ~2.5M rows
# Full corpus build time: ~15-20 minutes | Load from cache: ~10 seconds
# Full corpus can have peak memory during build: ~8-16GB

courts_index = bm25index.get_or_build_index(
    name="courts",
    csv_path=COURTS_CSV,
    index_path=COURTS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    max_rows=100000000  # Change to use bigger corpus
)
print(f"\nCourts index: {len(courts_index.documents):,} documents")

# Test search
test_results = courts_index.search("Meinungsfreiheit", top_k=3)
print(f"\nTest search 'Meinungsfreiheit': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")

Loading cached courts index from /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed/courts_index.pkl
  Loaded 2,476,315 documents

Courts index: 2,476,315 documents


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Test search 'Meinungsfreiheit': 1 results
  Top result: N/A


In [6]:
import re

In [7]:
from FlagEmbedding import FlagReranker
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-gemma', use_fp16=True)
#reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

`config.hidden_act` is ignored, you should use `config.hidden_activation` instead.
Gemma's activation function will be set to `gelu_pytorch_tanh`. Please, use
`config.hidden_activation` if you want to override this behaviour.
See https://github.com/huggingface/transformers/pull/29402 for more details.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of GemmaForSequenceClassification were not initialized from the model checkpoint at /root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-gemma and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
import query_by_dense
import citation_utils

court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

test_df = pd.read_csv('../data/test_rewrite_002.csv')

print("data loaded")

data loaded


In [ ]:
import rerank_utils
import rrf

query_id_2_hit_set_d = {}
query_id_2_query_list_d = {}
for query_id, q in zip(test_df['query_id'].tolist(), test_df['query'].tolist()):
    if query_id not in query_id_2_query_list_d:
        query_id_2_query_list_d[query_id] = []
    query_id_2_query_list = query_id_2_query_list_d[query_id]
    query_id_2_query_list.append(q)
    
    if query_id not in query_id_2_hit_set_d:
        query_id_2_hit_set_d[query_id] = set()
    query_id_2_hit_set = query_id_2_hit_set_d[query_id]

    test_results = courts_index.search(q, top_k=1000)[0]['hits']
    # test_results.extend(laws_index.search(q, top_k=100)[0]['hits'])
    
    for hit in test_results:
        query_id_2_hit_set.add(hit['citation'])
print("query_id_2_hit_set_d.keys:", query_id_2_hit_set_d.keys())

del q

query_id_l = []
citation_l = []
for query_id, query_id_2_hit_set in query_id_2_hit_set_d.items():
    first_layer_citation = []
    for hit in query_id_2_hit_set:
        first_layer_citation.append(hit)
        
    print("query_id:", query_id, "first_layer_citation.len:", len(first_layer_citation))

    raw_hits = citation_utils.BFS_citation(court_consideration_d, law_d, first_layer_citation, max_level=3) # 广度优先搜索

    print("raw_hits.len:", len(raw_hits))
    
    court_hits = [hits for hits in raw_hits if hits['citation'] in court_consideration_d]
    law_hits = [hits for hits in raw_hits if hits['citation'] in law_d]

    citations_court = []
    citations_law = []

    all_court_l = []
    for q in query_id_2_query_list_d[query_id]:
        __court_l = rerank_utils.rerank_by_dense_batch_with_score(reranker, q, court_hits, 10000)
        all_court_l.append([court[0]['citation'] for court in __court_l])
    final_citations_court = set([term[0] for term in rrf.compute(all_court_l)[:20]])

    all_law_l = []
    for q in query_id_2_query_list_d[query_id]:
        __law_l = rerank_utils.rerank_by_dense_batch_with_score(reranker, q, law_hits, 10000)
        all_law_l.append([law[0]['citation'] for law in __law_l])
    final_citations_law = set([term[0] for term in rrf.compute(all_law_l)[:20]])
    
    # for q in query_id_2_query_list_d[query_id]:
    #     __court_l = rerank_utils.rerank_by_dense_batch_with_score(reranker, q, court_hits, 10000)
    #     for court in __court_l:
    #         citations_court.append((court[0]['citation'], court[1]))
    
    #     __law_l = rerank_utils.rerank_by_dense_batch_with_score(reranker, q, law_hits, 10000)
    #     for law in __law_l:
    #         citations_law.append((law[0]['citation'], law[1]))

    # final_citations_court = set()
    # for c in sorted(citations_court, key=lambda x: x[1], reverse=True):
    #     if len(final_citations_court) >= 20:
    #         break
    #     if c[0] in final_citations_court:
    #         continue
    #     final_citations_court.add(c[0])

    # final_citations_law = set()
    # for c in sorted(citations_law, key=lambda x: x[1], reverse=True):
    #     if len(final_citations_law) >= 20:
    #         break
    #     if c[0] in final_citations_law:
    #         continue
    #     final_citations_law.add(c[0])
        

    citations = list(final_citations_court | final_citations_law)
    citation_l.append(';'.join(citations))
    query_id_l.append(query_id)

l = [(query_id, citation) for query_id, citation in zip(query_id_l, citation_l)]
l = sorted(l, key=lambda x: x[0])

new_id_l = [term[0] for term in l]
new_pc_l = [term[1] for term in l]

result_df = pd.DataFrame({'query_id':new_id_l, 'predicted_citations':new_pc_l})
result_df.to_csv("../data/result.csv", index=False)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

...

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

query_id_2_hit_set_d.keys: dict_keys(['test_001', 'test_002', 'test_003', 'test_004', 'test_005', 'test_006', 'test_007', 'test_008', 'test_009', 'test_010', 'test_011', 'test_012', 'test_013', 'test_014', 'test_015', 'test_016', 'test_017', 'test_018', 'test_019', 'test_020', 'test_021', 'test_022', 'test_023', 'test_024', 'test_025', 'test_026', 'test_027', 'test_028', 'test_029', 'test_030', 'test_031', 'test_032', 'test_033', 'test_034', 'test_035', 'test_036', 'test_037', 'test_038', 'test_039', 'test_040'])
query_id: test_001 first_layer_citation.len: 2541
raw_hits.len: 3196


rerank_by_dense:  70%|██████▉   | 1880/2701 [02:18<01:00, 13.47it/s]